# SRFM Quickstart

**Special Relativity in Financial Modeling**  
Version 1.1.0

This notebook demonstrates the core SRFM Python API:

1. Spacetime interval classification (TIMELIKE / LIGHTLIKE / SPACELIKE)
2. Lorentz factor computation
3. Relativistic backtest and performance comparison
4. Multi-asset spacetime with rolling correlation metric
5. Portfolio geodesic computation

## Installation

```bash
# Pure-Python (no build required):
pip install -e path/to/Special-Relativity-in-Financial-Modeling/python/

# With C++ extension:
pip install pybind11
cmake -B build -DSRFM_BUILD_PYTHON=ON
cmake --build build
pip install -e python/
```

In [ ]:
# Install if not present
import subprocess, sys
try:
    import srfm
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '../python/'])
    import srfm

print(f'SRFM version: {srfm.__version__}')
print(f'Speed of light (c): {srfm.SPEED_OF_LIGHT}')
print(f'Beta max safe: {srfm.BETA_MAX_SAFE}')

## 1. Spacetime Interval Classification

The core formula is:

$$ds^2 = -c^2 \Delta t^2 + \Delta P^2 + \Delta V^2 + \Delta M^2$$

| Sign | Regime | Market interpretation |
|------|--------|----------------------|
| $ds^2 < 0$ | TIMELIKE | Causal, momentum predictive — **1.27× lower next-bar variance** |
| $ds^2 = 0$ | LIGHTLIKE | Critical boundary |
| $ds^2 > 0$ | SPACELIKE | Stochastic, decorrelated |

In [ ]:
from srfm import SpacetimeInterval

# Classify individual intervals
examples = [
    # (dt, dP, dV, dM, description)
    (1.0, 0.5,  0.1, 0.05, 'Slow price move — causal'),
    (1.0, 1.5,  0.8, 0.3,  'Fast price move — stochastic'),
    (1.0, 1.0,  0.0, 0.0,  'Exactly light-cone (c=1)'),
    (0.5, 0.3,  0.2, 0.1,  'Short time, moderate move'),
    (2.0, 0.1,  0.0, 0.0,  'Long quiet bar'),
]

print(f"{'dt':>6} {'dP':>6} {'dV':>6} {'dM':>6} {'ds²':>10} {'Class':>12}  Description")
print('-' * 80)
for dt, dp, dv, dm, desc in examples:
    ds2 = SpacetimeInterval.compute_ds2(dt, dp, dv, dm)
    cls = SpacetimeInterval.classify(dt, dp, dv, dm)
    print(f"{dt:>6.1f} {dp:>6.2f} {dv:>6.2f} {dm:>6.2f} {ds2:>10.4f} {cls:>12}  {desc}")

## 2. Lorentz Factor

The Lorentz factor amplifies signals in slow-moving (causal) markets and
suppresses them in fast-moving (stochastic) regimes:

$$\gamma(\beta) = \frac{1}{\sqrt{1 - \beta^2}}, \quad \beta = \frac{\Delta P}{c \cdot \Delta t}$$

In [ ]:
import math
from srfm import LorentzTransform

betas = [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99]
print(f"{'β':>8} {'γ(β)':>10} {'Effect on signal'}")
print('-' * 50)
for b in betas:
    g = LorentzTransform.gamma(b)
    effect = f'{g:.3f}× signal weight'
    print(f"{b:>8.2f} {g:>10.4f}  {effect}")

# Also compute beta from price move
print()
print('β from price move: ΔP=0.5, Δt=1.0, c=1.0 →', LorentzTransform.beta(0.5, 1.0))
print('Relativistic momentum: γ=1.5, m_eff=1.0, p_raw=0.8 →',
      LorentzTransform.relativistic_momentum(1.5, 1.0, 0.8))

## 3. Relativistic Backtest

The backtester applies the Lorentz factor as a position multiplier:

$$\text{adjusted\_return}_t = \gamma(\beta_t) \cdot \text{raw\_return}_t$$

This up-weights signals from slow, causal markets and down-weights
signals from fast, stochastic markets.

In [ ]:
import random
from srfm import Backtester, BacktestConfig

# Generate a synthetic price series (random walk with drift)
random.seed(42)
prices = [100.0]
for _ in range(252):  # one trading year
    daily_ret = random.gauss(0.0002, 0.015)  # 2bp drift, 1.5% vol
    prices.append(prices[-1] * math.exp(daily_ret))

# Run backtest
cfg = BacktestConfig()
cfg.max_gamma = 3.0
cfg.annualisation = 252.0

bt = Backtester(cfg)
result = bt.run(prices)

print('=== Backtest Results ===')
print(result.to_string())
print()
print(f'Sharpe lift:    {result.sharpe_lift():+.4f}')
print(f'Sortino lift:   {result.sortino_lift():+.4f}')
print(f'Drawdown delta: {result.drawdown_delta():+.4f}  (positive = improvement)')
print(f'Mean γ:         {result.mean_gamma:.4f}')

## 4. Empirical Result Replication

The Q1 2025 empirical result: **TIMELIKE bars show 1.27× lower next-bar
absolute return variance** across 10 liquid instruments.

We can replicate this classification logic using the pure-Python API:

In [ ]:
from srfm import SpacetimeInterval
import math
import random

random.seed(2025)

# Simulate 1000 bars
prices = [100.0]
volumes = [1_000_000.0]
for _ in range(1000):
    p = prices[-1] * math.exp(random.gauss(0.0, 0.01))
    v = volumes[-1] * math.exp(random.gauss(0.0, 0.3))
    prices.append(p)
    volumes.append(v)

timelike_next_rets  = []
spacelike_next_rets = []

for i in range(1, len(prices) - 1):
    dt = 1.0
    dp = prices[i] - prices[i-1]
    dv = volumes[i] ** 0.25 - volumes[i-1] ** 0.25
    dm = (prices[i] - prices[i-1]) - (prices[i-1] - prices[i-2]) if i >= 2 else 0.0
    cls = SpacetimeInterval.classify(dt, dp, dv, dm)
    next_abs_ret = abs(math.log(prices[i+1] / prices[i]))
    if cls == 'TIMELIKE':
        timelike_next_rets.append(next_abs_ret)
    else:
        spacelike_next_rets.append(next_abs_ret)

def variance(xs):
    if len(xs) < 2:
        return 0.0
    mean = sum(xs) / len(xs)
    return sum((x - mean) ** 2 for x in xs) / (len(xs) - 1)

var_tl = variance(timelike_next_rets)
var_sl = variance(spacelike_next_rets)
ratio  = var_sl / var_tl if var_tl > 0 else float('nan')

print(f'TIMELIKE  bars: {len(timelike_next_rets):>5}  next-bar var: {var_tl:.6f}')
print(f'SPACELIKE bars: {len(spacelike_next_rets):>5}  next-bar var: {var_sl:.6f}')
print(f'Variance ratio (SL/TL): {ratio:.3f}×  (paper: 1.27×)')

## 5. Multi-Asset Spacetime

The multi-asset extension handles N correlated assets simultaneously,
using a rolling correlation matrix as the spatial metric block.

In [ ]:
from srfm import (
    MultiAssetEvent, CorrelationMetric, CorrelationMetricConfig,
    MultiAssetInterval, MultiAssetLorentz
)
import math, random

random.seed(42)
N = 3  # 3 assets: SPY, QQQ, IWM
symbols = ['SPY', 'QQQ', 'IWM']

# Simulate correlated price series
prices_series = [[100.0, 150.0, 80.0]]
for _ in range(120):
    market_shock = random.gauss(0.0, 0.008)
    row = []
    for p in prices_series[-1]:
        idio = random.gauss(0.0, 0.004)
        row.append(p * math.exp(market_shock + idio))
    prices_series.append(row)

# Build rolling correlation metric
cfg = CorrelationMetricConfig()
cfg.window_size = 60
cfg.c_market = 1.0
cm = CorrelationMetric(N, cfg)

# Feed price history
metric = None
for row in prices_series:
    m = cm.update(row)
    if m is not None:
        metric = m

print(f'Observations: {cm.observation_count()}')
print(f'Metric shape: ({len(metric)} × {len(metric[0])})')
print(f'g_00 (time-time): {metric[0][0]:.4f}')

corr = cm.correlation_matrix()
if corr:
    print(f'\nRolling correlation matrix ({N}×{N}):')
    for i, sym_i in enumerate(symbols):
        row_str = '  '.join(f'{corr[i][j]:>7.4f}' for j in range(N))
        print(f'  {sym_i}: [{row_str}]')

In [ ]:
# Classify the last interval
ev_a = MultiAssetEvent()
ev_a.symbols   = symbols
ev_a.prices    = prices_series[-2]
ev_a.volumes   = [1e6] * N
ev_a.timestamp = 1_735_000_000_000  # arbitrary epoch ms

ev_b = MultiAssetEvent()
ev_b.symbols   = symbols
ev_b.prices    = prices_series[-1]
ev_b.volumes   = [1e6] * N
ev_b.timestamp = ev_a.timestamp + 1000  # 1 second later

if metric:
    ds2 = MultiAssetInterval.compute(ev_a, ev_b, metric)
    if ds2 is not None:
        cls = MultiAssetInterval.classify(ds2)
        print(f'Multi-asset interval ds² = {ds2:.6f} → {cls}')

    # Multi-asset Lorentz transform
    result = MultiAssetLorentz.transform(ev_a, ev_b, metric)
    if result:
        print(f'Portfolio β:   {result.beta_portfolio:.4f}')
        print(f'Portfolio γ:   {result.gamma_portfolio:.4f}')
        for i, sym in enumerate(symbols):
            print(f'  {sym}: β={result.per_asset_beta[i]:>7.4f}  '
                  f'γ={result.per_asset_gamma[i]:>7.4f}  '
                  f'adj_price_Δ={result.adjusted_prices[i]:>9.4f}')

## 6. Portfolio Geodesic

In multi-asset spacetime, the geodesic is the "natural" portfolio
trajectory.  Deviations of actual prices from the geodesic are signals.

In [ ]:
from srfm import PortfolioGeodesic

if metric:
    # Four-velocity: unit time-direction + price trend from last 5 bars
    trend = [(prices_series[-1][i] - prices_series[-6][i]) / 5.0
             for i in range(N)]
    four_velocity = [1.0] + trend  # (dt/dτ, dP_1/dτ, dP_2/dτ, dP_3/dτ)

    # Integrate 10 steps forward
    path = PortfolioGeodesic.integrate(
        initial      = ev_b,
        four_velocity= four_velocity,
        metric       = metric,
        n_steps      = 10,
        dt           = 1.0,
    )

    print(f'Geodesic path ({len(path)} steps):')
    print(f"{'Step':>5} {'τ':>8} {'arc_len':>10}  {'Prices'}")
    print('-' * 60)
    for k, step in enumerate(path):
        p_str = '  '.join(f'{p:>9.3f}' for p in step.event.prices)
        print(f"{k+1:>5} {step.proper_time:>8.3f} {step.path_length:>10.4f}  [{p_str}]")

    # Portfolio weights along the geodesic
    weights = PortfolioGeodesic.portfolio_weights(path, four_velocity, gross_exposure=1.0)
    print(f'\nGeodesic portfolio weights at step 1: {[f"{w:.4f}" for w in weights[0]]}')

## Summary

This notebook demonstrated:

1. **SpacetimeInterval.classify()** — classifies OHLCV bars as TIMELIKE (causal) or SPACELIKE (stochastic) using the Minkowski metric.
2. **LorentzTransform.gamma()** — computes γ(β) for use as a signal weight multiplier.
3. **Backtester.run()** — runs a full relativistic backtest returning Sharpe, Sortino, max drawdown, and relativistic lift.
4. **CorrelationMetric** — builds a rolling Lorentzian metric tensor from the covariance matrix of N assets.
5. **MultiAssetInterval** — classifies joint portfolio movements in N-dimensional spacetime.
6. **MultiAssetLorentz** — applies γ-scaling across correlated asset price changes.
7. **PortfolioGeodesic** — computes the inertial portfolio trajectory; deviations are trading signals.

### Next Steps
- See the [paper](../paper/) for full mathematical derivations.
- See [BENCHMARKS.md](../BENCHMARKS.md) for performance measurements.
- See [BUILD.md](../BUILD.md) for building the C++ extension for maximum performance.